In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna

import numpy as np
import pandas as pd
import torch

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.bigru_wrapper import BiGRUModel
from src.trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "BiGRU"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


## Introducción

La Unidad Recurrente con Compuertas Bidireccional (*Bidirectional Gated Recurrent Unit*, BiGRU) extiende la arquitectura GRU estándar procesando cada ventana temporal en dos direcciones: hacia adelante (de los rezagos más antiguos hacia el instante más reciente) y hacia atrás (desde el extremo más reciente de la ventana hacia los rezagos anteriores). Al combinar ambos recorridos, la red construye representaciones ocultas que integran el contexto de la secuencia completa en cada paso antes de proyectar la varianza condicional.

En el pronóstico de volatilidad del crudo Brent bajo el protocolo del *benchmark*, la BiGRU permite capturar relaciones no lineales entre los retornos históricos y la varianza futura sin imponer una forma paramétrica previa. El objetivo de esta sección es presentar, entrenar y evaluar un modelo BiGRU manteniendo las mismas condiciones de información, preprocesamiento y validación que en el resto de modelos del proyecto.

### Arquitectura

Las ecuaciones internas de las compuertas de actualización $\left(\mathbf{z}_t\right)$, compuerta de reinicio $\left(\mathbf{r}_t\right)$, estado candidato ($\tilde{\mathbf{h}}_t$) y actualización del estado oculto son iguales que en el GRU estándar, aplicadas de forma independiente a cada dirección temporal (directa e inversa).

A diferencia del GRU unidireccional, para una secuencia de longitud $L$, la capa BiGRU produce estados ocultos en ambos sentidos y toma los estados finales de la última capa en cada dirección, $\mathbf{h}_t^{\rightarrow} \in \mathbb{R}^{H}$ y $\mathbf{h}_t^{\leftarrow} \in \mathbb{R}^{H}$ (donde $H$ es el tamaño oculto `hidden_size`), concatenándolos en un vector de dimensión $2H$:

$$
\mathbf{h}_t^{\text{bi}} =
\begin{bmatrix}
\mathbf{h}_t^{\rightarrow} \\
\mathbf{h}_t^{\leftarrow}
\end{bmatrix}
\in \mathbb{R}^{2H}.
$$

Ese vector concatenado pasa por una capa lineal con pesos $\mathbf{w} \in \mathbb{R}^{2H}$ y sesgo $b \in \mathbb{R}$ para obtener la salida escalar previa:

$$
y_t = \mathbf{w}^{\top}\mathbf{h}_t^{\text{bi}} + b,
$$

y luego por la activación *Softplus* para garantizar una varianza estrictamente positiva:

$$
\hat{\sigma}_t^2 = \text{Softplus}(y_t) + \varepsilon = \ln\left(1 + e^{y_t}\right) + \varepsilon,
$$

donde $\varepsilon$ es una pequeña constante numérica positiva (`1e-8`) y $\boldsymbol{\theta}$ agrupa los parámetros entrenables de la red.

La estandarización Z sobre los retornos logarítmicos diarios (ajustada exclusivamente sobre el 80 % inicial de cada fold de entrenamiento para evitar fuga de información), la definición de la ventana de entrada $\mathbf{Z}_t \in \mathbb{R}^{L \times 1}$, la estrategia one-step-ahead minimizando la pérdida QLIKE frente a la varianza de Rogers-Satchell $\left(RS_i\right)$ y el entrenamiento con retropropagación y optimizador Adam siguen el mismo esquema que en el GRU. En la implementación en `PyTorch`, en cambio, la red se define mediante `BiGRUNetwork` con `bidirectional=True`, concatena los estados finales de ambas direcciones (`h_n[-2]` y `h_n[-1]`) y proyecta un vector de dimensión $2H$ hacia la salida con `Softplus`.

## Entrenamiento

Para la selección de hiperparámetros del modelo BiGRU, se mantuvo el mismo protocolo experimental Walk-Forward y el mismo presupuesto computacional que en los modelos paramétricos y en la GRU. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la arquitectura de la red neuronal y sus parámetros de regularización sobre exactamente el mismo espacio de búsqueda que la GRU.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos para cada ventana del esquema Walk-Forward. Siguiendo los principios de validación cruzada para series temporales, cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura durante el entrenamiento.

El espacio de búsqueda incluyó exactamente los mismos hiperparámetros que en la GRU: la longitud de la ventana de entrada (`look_back`), el número de neuronas en la capa oculta (`hidden_size`), la cantidad de capas recurrentes, la tasa de abandono (`dropout`) y la tasa de aprendizaje. Para mitigar el sobreajuste, se incorporó también la penalización $\ell_2$ (`weight_decay`). El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def bigru_search_space(trial: optuna.Trial) -> dict:
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128]),
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=BiGRUModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=bigru_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

In [3]:
import time

scorer = VolatilityScorer()

best_model = BiGRUModel(
    look_back=14,
    hidden_size=64,
    num_layers=1,
    dropout=0.2,
    learning_rate=1e-3,
    weight_decay=1e-6,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    fine_tune_epochs=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time
print(f">> fit OK en {elapsed_time:.2f} s")
print("   diagnostics:", best_model.get_diagnostics())

n_oos = min(2, len(y_test))
y_test_20 = y_test[-n_oos:]
rs_test_20 = rs_test[-n_oos:]

t0_oos = time.time()
preds_test_20 = best_model.forecast(y_test_20, rs_seq=rs_test_20, debug=False)
oos_time = time.time() - t0_oos

print(
    f">> forecast (últimos {n_oos} días, fine_tune_epochs=1) OK en {oos_time:.2f} s "
    f"| shape={preds_test_20.shape}"
)

>> fit OK en 19.08 s
   diagnostics: {'converged': True, 'epochs_run': 26}
>> forecast (últimos 2 días, fine_tune_epochs=1) OK en 48.25 s | shape=(2,)


In [4]:
import numpy as np
from src.utils import trainer_toolkit as tt

history = best_model.get_history()
diagnostics = best_model.get_diagnostics()

train_loss = np.asarray(history.get("train_loss", []), dtype=float)
val_loss = np.asarray(history.get("val_loss", []), dtype=float)

assert len(train_loss) > 0 and len(val_loss) > 0, "El historial de entrenamiento está vacío."
assert np.isfinite(train_loss).all(), "train_loss contiene NaN o Inf."
assert np.isfinite(val_loss).all(), "val_loss contiene NaN o Inf."

print(
    f"Épocas ejecutadas: {diagnostics.get('epochs_run', len(train_loss))} "
    f"| Convergió: {diagnostics.get('converged')}"
)
print(
    f"Train QLIKE inicial -> final/mínimo: {train_loss[0]:.6f} -> {train_loss[-1]:.6f} "
    f"(min={train_loss.min():.6f})"
)
print(
    f"Val   QLIKE inicial -> final/mínimo: {val_loss[0]:.6f} -> {val_loss[-1]:.6f} "
    f"(min={val_loss.min():.6f})"
)

max_epochs_plotted = len(train_loss)

tt.plot_learning_curve(
    history=history,
    title="Curva de Aprendizaje (BiGRU)",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14,
    ylab_pad=14,
    line_width=2.0,
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
    ylim=(0, 5),
    xlim=(0, max(10, max_epochs_plotted)),
    y_step=0.5,
    x_step=5,
    legend_bbox=(0.95, 0.95),
)

Épocas ejecutadas: 26 | Convergió: True
Train QLIKE inicial -> final/mínimo: 6.775556 -> 0.790881 (min=0.790881)
Val   QLIKE inicial -> final/mínimo: 6.072702 -> 0.555004 (min=0.497959)


In [5]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from src.utils import test_toolkit as ts

best_model.net.eval()
look_back = best_model.context_length

# Escalar y_train igual que en fit() y armar ventanas de longitud look_back
y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()
rs_train_scaled_target = rs_train * best_model.target_scale

x_windows = np.lib.stride_tricks.sliding_window_view(y_train_scaled, window_shape=look_back)
x_windows = x_windows[:-1, :, None].astype(np.float32)  # (N - look_back, look_back, 1)
y_targets = rs_train_scaled_target[look_back:].astype(np.float32)

train_eval_dataset = TensorDataset(
    torch.tensor(x_windows, dtype=torch.float32),
    torch.tensor(y_targets, dtype=torch.float32),
)
train_eval_loader = DataLoader(
    train_eval_dataset, batch_size=best_model.batch_size, shuffle=False
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_model.device)
        preds_batch = best_model.net(x_batch).squeeze(-1).cpu().numpy()
        preds_train_list.append(preds_batch)

# Revertir target_scale para quedar en varianza original
preds_train_var = np.concatenate(preds_train_list) / best_model.target_scale
n_preds = len(preds_train_var)

dates_aligned = df_train.index[look_back : look_back + n_preds]
rs_volatilidad_aligned = np.sqrt(
    np.maximum(rs_train[look_back : look_back + n_preds], 0.0)
)
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (BiGRU)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad BiGRU $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)